# Lean4 PPO — Kaggle training

Two stages:

1. **BC initialization** on `lean4_bc_dataset.npz` (existing replay corpus).
2. **Native PPO self-play** using Dipam Chakraborty's public PPO trainer, with the lean4 deck added to the deck pool.

Attach:
- this notebook's code/repo as a dataset or clone it below,
- the PPO training repo,
- the competition data (`cg/` and `EN_Card_Data_R2_full.csv`),
- `lean4_ppo_bundle.zip` (BC dataset + lean4 deck).

Training is compute-heavy. Start with a short run, inspect telemetry, then scale.


In [ ]:
# 0) Locate inputs
from pathlib import Path
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
WORK.mkdir(exist_ok=True)
print("input dirs:", [p.name for p in INPUT.iterdir()][:20])


In [ ]:
# 1) Get the PPO trainer (github). If internet is off, attach the repo as a dataset instead.
REPO = WORK / "ptcg_ppo"
if not (REPO / "native/train_native.py").exists():
    !git clone --depth 1 https://github.com/dipamc/kaggle-ptcg-ai-battle.git {REPO}
print(REPO)


In [ ]:
# 2) Locate engine + card table from either the competition input or lean4_full_bundle.zip.
import shutil
from pathlib import Path

cg_game = next(INPUT.rglob("cg/game.py"), None)
assert cg_game is not None, "cg/game.py not found: attach competition data or lean4_full_bundle.zip"
cg_dir = cg_game.parent
card_csv = next(INPUT.rglob("EN_Card_Data_R2_full.csv"), None)
assert card_csv is not None, "EN_Card_Data_R2_full.csv not found: attach competition data or lean4_full_bundle.zip"
print("cg:", cg_dir)
print("card:", card_csv)
shutil.copytree(cg_dir, REPO/"data/cg", dirs_exist_ok=True)
shutil.copy2(card_csv, REPO/"data/EN_Card_Data.csv")
# New-card embedding tables must match the 1448-card pool.
embed = next(INPUT.rglob("embed_pca.npz"), None)
assert embed is not None, "embed_pca.npz not found: attach lean4_full_bundle.zip"
shutil.copy2(embed, REPO/"data/embed_pca.npz")
ling = next(INPUT.rglob("lingering_effects.json"), None)
if ling is not None:
    shutil.copy2(ling, REPO/"data/lingering_effects.json")
print("embed:", embed)


In [ ]:
# 3) Build a lean4 deck pool. Use the public pool plus the lean4 list.
import shutil
from pathlib import Path
pool = REPO/"decks/pool_lean4"
shutil.rmtree(pool, ignore_errors=True)
shutil.copytree(REPO/"decks/pool", pool)
# lean4 list: include in the bundle or paste it
lean4 = WORK/"lean4_deck.csv"
if not lean4.exists():
    lean4.write_text("""2
2
2
2
2
5
5
5
5
5
119
119
119
119
120
120
120
120
121
121
121
140
235
235
235
1071
1079
1080
1086
1086
1086
1086
1097
1097
1120
1120
1120
1120
1121
1121
1121
1121
1152
1152
1152
1152
1182
1182
1182
1198
1198
1198
1213
1213
1227
1227
1227
1227
1256
1256
""")
shutil.copy2(lean4, pool/"lean4.csv")
print("pool decks:", len(list(pool.glob('*.csv'))))


In [ ]:
# 4) Build tables + native trainer. Patch the trainer to the current card pool.
import os, subprocess, glob
from pathlib import Path
os.chdir(REPO)

def patch(path, pairs):
    p = Path(path); s = p.read_text()
    for old, new in pairs:
        if old in s:
            s = s.replace(old, new)
        elif new not in s:
            raise AssertionError(f"{path}: missing {old!r} (and already-patched marker absent)")
    p.write_text(s)
    print("patched", path)

# Current Playground pool: 1431 cards / 1755 attacks / 470 skills,
# plus the trainer's 17/17/27 null/slack rows.
patch("ptcg/rl/buffers.py", [
    ("N_CARDS = 1267 + 17", "N_CARDS = 1431 + 17"),
    ("N_ATTACKS = 1556 + 17", "N_ATTACKS = 1755 + 17"),
])
patch("ptcg/rl/cards.py", [
    ("N_SKILLS = 421 + 27", "N_SKILLS = 470 + 27"),
])
patch("native/ptcg/ptcg_env.h", [
    ("#define PT_N_CARDS (1267 + 17)", "#define PT_N_CARDS (1431 + 17)"),
    ("#define PT_N_ATTACKS (1556 + 17)", "#define PT_N_ATTACKS (1755 + 17)"),
    ("#define PT_N_SKILLS (421 + 27)", "#define PT_N_SKILLS (470 + 27)"),
])
patch("native/tools/native_weights.py", [
    ("NC, NA, NS = 1284, 1573, 448", "NC, NA, NS = 1448, 1772, 497"),
])
patch("native/model/ptcg_model.cu", [
    ("#define PT_NC 1284", "#define PT_NC 1448"),
    ("#define PT_NA 1573", "#define PT_NA 1772"),
    ("#define PT_NS 448", "#define PT_NS 497"),
])

# Kaggle has libnvidia-ml.so.1 but often lacks the -dev .so symlink.
cands = []
for pat in ("/usr/lib/x86_64-linux-gnu/libnvidia-ml.so*",
            "/usr/local/nvidia/lib64/libnvidia-ml.so*",
            "/usr/lib64/libnvidia-ml.so*"):
    cands += glob.glob(pat)
lib = next((p for p in cands if p.endswith(".so.1")), cands[0] if cands else None)
assert lib, "libnvidia-ml.so.1 not found"
nvml = Path("/kaggle/working/nvml"); nvml.mkdir(parents=True, exist_ok=True)
link = nvml / "libnvidia-ml.so"
if not link.exists(): link.symlink_to(lib)
bs = Path("native/build_native.sh"); text = bs.read_text()
if 'NVML_LFLAG=""' in text:
    text = text.replace('NVML_LFLAG=""', f'NVML_LFLAG="-L{nvml}"', 1)
    bs.write_text(text)
print("nvml:", lib, "->", link)

def run(cmd, env_extra=None):
    env = {**os.environ, **(env_extra or {})}
    r = subprocess.run(cmd, env=env, text=True, capture_output=True)
    print("\n$", " ".join(cmd), "rc=", r.returncode)
    if r.stdout: print(r.stdout[-4000:])
    if r.stderr: print("STDERR:", r.stderr[-4000:])
    if r.returncode != 0:
        raise RuntimeError(f"command failed: {cmd}")
    return r

run(["python", "tools/build_carddb.py"], {"PYTHONPATH": "data"})
run(["python", "native/tools/export_tables.py", "native/ptcg_tables.bin",
     "--pool", "decks/pool_lean4", "--alt-pool", "decks/pool_alt"],
    {"PYTHONPATH": "data:."})
run(["bash", "native/build_native.sh"])


In [ ]:
# 5) Train against the frozen public PPO league bank.
# Stop the old run first; use a NEW run name so it starts with the league bank.
import os, subprocess
from pathlib import Path
league = next(INPUT.rglob("ppo_public_native.bin"), None)
assert league is not None, "ppo_public_native.bin not found: attach lean4_full_bundle.zip"
import shutil
run_dir = Path("experiments/native-lean4_ppo_league49_init")
run_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(league, run_dir/"init.bin")
print("init from public PPO:", league, "->", run_dir/"init.bin")
cmd = [
 "python","native/train_native.py",
 "--run-name","lean4_ppo_league49_init",
 "--deck-pool","decks/pool_lean4",
 "--gpus","1",
 "--total-agents","512",
 "--minibatch","256",
 "--timesteps","20000000",
 "--checkpoint-interval","10",
 "--mix-self","1.0",
 "--anchor-decks","lean4",
 "--deck-matrix-every","20000",
 "--league-ckpts", str(league),
 "--league-pct","0.49",
]
env={**os.environ, "PYTHONPATH":"data:.:native", "PTCG_TABLES":"native/ptcg_tables.bin"}
print(" ".join(cmd))
proc=subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line,end="",flush=True)
print("exit",proc.wait())


In [ ]:
# 6) Convert a checkpoint to a torch model for evaluation / submission.
# Replace the step with one from experiments/native-lean4_ppo/
import subprocess, os, glob
bins = sorted(glob.glob(str(WORK/"ptcg_ppo/experiments/native-lean4_ppo/*.bin")))
print("checkpoints:", bins[:5])
# Example: convert final.bin to model.pt
# subprocess.run(["python","native/tools/native_weights.py","import",
#                 "experiments/native-lean4_ppo/final.bin","/kaggle/working/lean4_ppo.pt"],
#                env={**os.environ,"PYTHONPATH":"data:."})
